# Projeto Fictus | Análise Financeira — Bloco 2: Inadimplência e Exposição ao Risco

---

## Pergunta Central do Bloco
> **O risco de crédito da empresa-alvo é estruturalmente controlado — ou apresenta volatilidade que representa risco relevante para o comprador?**

---

## Contexto do Bloco

O Bloco 1 mapeou a estrutura de recebimento: o mix de pagamento, o prazo médio, o capital em aberto e o spread capturado por intermediários. A predominância de crédito parcelado observada no Bloco 1 cria exposição direta ao risco tratado aqui: **o risco de crédito latente na operação é gerenciável — ou representa uma fragilidade estrutural que o comprador herdaria?**

O dataset Olist não registra inadimplência real — não há dados de chargeback, atraso de pagamento ou calote. O que existe são **proxies analíticos de risco**: pedidos cancelados, pagamentos não definidos e pedidos não entregues que funcionam como sinais de risco latente. Esta análise trabalha com esses proxies de forma declarada e transparente.

**Este bloco investiga:**
1. Distribuição do Risco: Concentração ou Difuso?
2. Mapa Tridimensional de Risco: Ticket × Prazo × Região
3. O Risco Cresce com o Volume?
4. Distribuição Estatística do Risco: É Precificável?
5. Análise de Sensibilidade: Impacto de Choques de Risco

---

## Nota Metodológica
As métricas de inadimplência neste bloco são **proxies analíticos**, não dados reais de crédito.
`anomalia_pagamento` = pedido cancelado, unavailable ou com tipo_pagamento = nao_definido.
Esta limitação é declarada explicitamente — a análise é válida como diagnóstico de risco latente,
não como mensuração de inadimplência real.

---


## Configuração e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_FIN     = BASE_DIR / "data" / "finance"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")

def ler(f, **kw):
    df = pd.read_csv(DIR_FIN / f, low_memory=False, **kw)
    df.columns = df.columns.str.strip()
    return df

fin_fato   = ler("fin_fato.csv")
fin_mensal = ler("fin_mensal.csv")
fin_trim   = ler("fin_trimestral.csv")
fin_pag    = ler("fin_pagamento.csv")
fin_reg    = ler("fin_regional.csv")
fin_faixa  = ler("fin_faixa.csv")

for col in ["preco", "numero_parcelas", "pmr_ajustado", "spread_intermediario",
            "capital_em_aberto", "anomalia_pagamento"]:
    if col in fin_fato.columns:
        fin_fato[col] = pd.to_numeric(fin_fato[col], errors="coerce")

periodos_ord = sorted(fin_fato["periodo"].dropna().unique())
print(f"Dados: {len(fin_fato):,} registros | {periodos_ord[0]} a {periodos_ord[-1]}")
print(f"Taxa de anomalia geral: {fin_fato['anomalia_pagamento'].mean()*100:.2f}%")


---

## Análise 1 — Distribuição do Risco: Concentração ou Difuso?

> *"Risco de crédito raramente é distribuído uniformemente. A análise aplica o Princípio de Pareto para identificar se a exposição está concentrada em poucas categorias, regiões ou faixas de parcelamento — ou se é difusa por toda a operação. Risco concentrado é gerenciável e precificável; risco difuso é estrutural e representa exposição relevante para o comprador, pois não pode ser mitigado com ajustes pontuais de portfólio."*

**Framework:** Pareto — concentração do risco  
**Entrega:** Distribuição do risco por categoria, região e faixa de parcelamento

**Como este script responde à pergunta:**
> O script calcula a taxa de anomalia de pagamento agrupada por categoria, estado e faixa de parcelamento. Três painéis respondem à pergunta:
>1. **Top 15 categorias por anomalia (barras horizontais):** Categorias com taxa acima da mediana aparecem em vermelho; abaixo, em cinza. Se poucos segmentos concentram as barras vermelhas, o risco é concentrado — gerenciável. Se a maioria é vermelha, o risco é difuso — estrutural.
>2. **% de anomalia por estado (top 10 por volume):** Barras verticais com anotação do percentual, coloridas por nível relativo à mediana. Revela se o risco geográfico está disperso ou se há estados que concentram o problema.
>3. **% de anomalia por faixa de parcelamento:** Barras por faixa, coloridas por nível relativo à média. Responde se o risco cresce com o prazo de parcelamento — o que definiria quais segmentos nunca deveriam ser internalizados.

**Análise do Resultado:**
A distribuição do risco por categoria, região e prazo define o tipo de intervenção necessária pós-aquisição. Se o risco está concentrado em segmentos específicos, o comprador pode precificá-lo e negociá-lo no valuation. Se é difuso — presente em todas as categorias e regiões com intensidade semelhante — representa uma característica estrutural da base de clientes que exige mudança de política de crédito e não apenas ajuste de portfólio. O Pareto de anomalias é, portanto, o primeiro filtro para classificar o risco herdado como gerenciável ou estrutural.

In [ ]:
# ─── Concentração do risco por dimensão ──────────────────────────────────────
# Risco por categoria
risco_cat = (
    fin_fato.groupby("nome_categoria_produto")
    .agg(n_pedidos    = ("id_pedido",          "nunique"),
         n_anomalias  = ("anomalia_pagamento",  "sum"),
         receita      = ("preco",               "sum"),
         ticket_medio = ("preco",               "mean"))
    .reset_index()
)
risco_cat["pct_anomalia"] = risco_cat["n_anomalias"] / risco_cat["n_pedidos"] * 100
risco_cat["pct_receita"]  = risco_cat["receita"] / risco_cat["receita"].sum() * 100
risco_cat_top = risco_cat.nlargest(15, "n_anomalias")

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle("Bloco 2 — Distribuição e Concentração do Risco de Crédito (Proxy)",
             fontsize=13, fontweight="bold")

# Painel 1: Top categorias por anomalia
cores_cat = [COR_ALERTA if p > risco_cat["pct_anomalia"].median() else COR_NEUTRO
             for p in risco_cat_top["pct_anomalia"]]
bars = axes[0].barh(risco_cat_top["nome_categoria_produto"],
                    risco_cat_top["pct_anomalia"], color=cores_cat, alpha=0.85)
axes[0].set_title("Top 15 Categorias — % de Anomalias\n(proxy de risco)", fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].tick_params(axis="y", labelsize=7)

# Painel 2: Risco por região (top 10)
fin_reg_top = fin_reg.nlargest(10, "n_pedidos")
cores_reg = [COR_ALERTA if p > fin_reg["pct_anomalia"].median() else COR_RECEITA
             for p in fin_reg_top["pct_anomalia"]]
bars2 = axes[1].bar(fin_reg_top["estado_cliente"], fin_reg_top["pct_anomalia"],
                    color=cores_reg, alpha=0.85)
axes[1].set_title("% de Anomalias por Estado\n(top 10 por volume)", fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].tick_params(axis="x", rotation=45)
for bar, val in zip(bars2, fin_reg_top["pct_anomalia"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=8)

# Painel 3: Risco por faixa de parcelamento
ORDEM = ["a_vista", "2x_3x", "4x_6x", "7x_12x", "13x_mais"]
fin_faixa_ord = fin_faixa.set_index("faixa_parcelamento").reindex(
    [f for f in ORDEM if f in fin_faixa["faixa_parcelamento"].values]
).reset_index()
cores_faixa = [COR_ALERTA if p > fin_faixa["pct_anomalia"].mean() else COR_RECEITA
               for p in fin_faixa_ord["pct_anomalia"]]
bars3 = axes[2].bar(fin_faixa_ord["faixa_parcelamento"], fin_faixa_ord["pct_anomalia"],
                    color=cores_faixa, alpha=0.85)
axes[2].set_title("% de Anomalias por Faixa de Parcelamento", fontsize=9)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[2].tick_params(axis="x", rotation=20)
for bar, val in zip(bars3, fin_faixa_ord["pct_anomalia"]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
salvar(fig, "02_concentracao_risco")
plt.show()


---

## Análise 2 — Mapa Tridimensional de Risco: Ticket × Prazo × Região

> *"O trinômio ticket × prazo × localização define o perfil de exposição ao risco. Analisar as três dimensões simultaneamente evita o erro de mitigar risco em segmentos irrelevantes enquanto a maior concentração passa despercebida nas médias agregadas. O cruzamento dessas variáveis identifica quais combinações específicas de produto, prazo e geografia concentram o risco mais elevado — informação crítica para a avaliação da qualidade real da base de receita."*

**Framework:** Análise de risco multidimensional — cruzamento de variáveis de exposição  
**Entrega:** Mapa de risco multidimensional com heatmap de concentração

**Como este script responde à pergunta:**
> O script combina ticket, parcelamento e região em dois painéis complementares:
> 1. **Scatter ticket médio × % anomalia por categoria:** Cada bolha é uma categoria — o tamanho representa a participação na receita e a cor o percentual de anomalia (verde = baixo risco, vermelho = alto). Categorias no quadrante superior direito — alto ticket e alta anomalia — concentram receita relevante e risco elevado simultaneamente, representando a maior exposição potencial para o comprador.
> 2. **Heatmap região × faixa de parcelamento:** Matriz com os 10 estados de maior volume nas linhas e as faixas de parcelamento nas colunas. Cada célula mostra o percentual de anomalia e é colorida do verde (baixo) ao vermelho (alto). Células vermelhas no cruzamento de regiões críticas com parcelamentos longos são os segmentos de maior risco estrutural.

**Análise do Resultado:**
O quadrante de alto ticket e alta anomalia é o de maior criticidade para o comprador: concentra receita relevante e risco elevado simultaneamente. Categorias nesse quadrante não podem ser simplesmente descontinuadas sem impacto significativo no faturamento — exigem renegociação de condições de parcelamento ou ajuste de política de crédito por segmento. O heatmap região × prazo complementa esse diagnóstico ao revelar se o risco geográfico se intensifica em faixas de parcelamento mais longas — o sinal mais preocupante, pois indica que a expansão de prazos está sendo usada como alavanca comercial em regiões de maior inadimplência histórica.


In [ ]:
# ─── Mapa de risco: ticket × parcelamento × região ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Bloco 2 — Mapa de Risco: Ticket × Parcelamento × Região",
             fontsize=13, fontweight="bold")

# Painel 1: Scatter — ticket médio vs. % anomalia por categoria
scatter_data = risco_cat[risco_cat["n_pedidos"] >= 10].copy()
sc = axes[0].scatter(
    scatter_data["ticket_medio"],
    scatter_data["pct_anomalia"],
    s=scatter_data["pct_receita"] * 20,
    alpha=0.6,
    c=scatter_data["pct_anomalia"],
    cmap="RdYlGn_r",
    edgecolors="white",
    linewidths=0.5
)
plt.colorbar(sc, ax=axes[0], label="% Anomalia")
axes[0].set_xlabel("Ticket Médio (R$)")
axes[0].set_ylabel("% Anomalia de Pagamento")
axes[0].set_title("Risco por Categoria\n(tamanho = participação na receita)", fontsize=10)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))

# Painel 2: Heatmap região × faixa de parcelamento
hm_data = (
    fin_fato[fin_fato["estado_cliente"].isin(fin_reg.nlargest(10, "n_pedidos")["estado_cliente"])]
    .groupby(["estado_cliente", "faixa_parcelamento"])
    .agg(pct_anomalia=("anomalia_pagamento", "mean"))
    .reset_index()
    .pivot(index="estado_cliente", columns="faixa_parcelamento", values="pct_anomalia")
    .fillna(0) * 100
)
ORDEM_HM = [f for f in ["a_vista", "2x_3x", "4x_6x", "7x_12x", "13x_mais"]
            if f in hm_data.columns]
hm_data = hm_data[ORDEM_HM]
sns.heatmap(hm_data, ax=axes[1], cmap="RdYlGn_r", annot=True, fmt=".1f",
            linewidths=0.5, cbar_kws={"label": "% Anomalia"})
axes[1].set_title("Heatmap: % Anomalia por Região × Faixa de Parcelamento", fontsize=10)
axes[1].set_xlabel("Faixa de Parcelamento")
axes[1].set_ylabel("Estado do Cliente")

plt.tight_layout()
salvar(fig, "02_mapa_risco_tridimensional")
plt.show()

print("\n── Segmentos de Maior Risco ──────────────────────")
print("  Categorias com anomalia acima da média:")
cats_risco = risco_cat[risco_cat["pct_anomalia"] > risco_cat["pct_anomalia"].mean()]
cats_risco_ord = cats_risco.sort_values("pct_anomalia", ascending=False).head(10)
for _, row in cats_risco_ord.iterrows():
    print(f"    {row['nome_categoria_produto']:<40} {row['pct_anomalia']:.1f}%")


---

## Análise 3 — O Risco Cresce com o Volume?

> *"Em um portfólio bem diversificado, o crescimento de volume tende a diluir o risco individual por efeito de pulverização. Entretanto, se os novos clientes captados apresentam perfil de pagamento distinto dos clientes consolidados, o crescimento pode concentrar risco progressivamente. A correlação entre volume e taxa de anomalia determina se escalar o negócio amplifica ou atenua a exposição ao risco de crédito — uma das perguntas mais relevantes para um comprador que pretende acelerar o crescimento pós-aquisição"*

**Framework:** PDCA + Análise de tendência  
**Entrega:** Taxa de anomalias de pagamento versus volume com teste de correlação e tendência

**Como este script responde à pergunta:**
> O script rastreia a evolução da taxa de anomalia em paralelo com o volume e calcula a correlação entre as duas séries. Dois painéis respondem à pergunta:
>1. **Volume de pedidos × taxa de anomalia mensal (eixos duplos):** Barras de volume (eixo esquerdo) e linha de anomalia com linha de tendência (eixo direito). O slope da tendência, anotado na legenda, mostra se o risco está aumentando, estável ou diminuindo ao longo do tempo — independentemente do volume.
>2. **Scatter volume × % anomalia com anotação de mês:** Cada ponto é um mês, anotado com o período. A linha de regressão e o coeficiente de correlação no título respondem diretamente: meses de maior volume têm mais ou menos anomalia? Correlação positiva e significativa é o sinal mais preocupante — significa que crescer em pedidos aumenta proporcionalmente o risco de crédito.

**Análise do Resultado:**
Uma correlação positiva e estatisticamente significativa entre volume e taxa de anomalia é o sinal mais preocupante deste bloco: significa que a estratégia de crescimento atual está comprando risco de crédito de forma proporcional. Nesse cenário, acelerar o crescimento pós-aquisição sem revisar a política de concessão de parcelamento ampliaria a exposição financeira na mesma proporção. Se a correlação é negativa ou não significativa, o crescimento dilui o risco — e o comprador herda um modelo mais seguro à medida que o volume escala.


In [ ]:
# ─── Risco vs. volume: evolução temporal ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Bloco 2 — Comportamento do Risco com o Crescimento do Volume",
             fontsize=13, fontweight="bold")

# Painel 1: Volume vs. % anomalia ao longo do tempo
ax_vol = axes[0]
ax_risco = ax_vol.twinx()
ax_vol.bar(range(len(fin_mensal)), fin_mensal["n_pedidos"],
           color=COR_RECEITA, alpha=0.4, label="Volume de Pedidos")
ax_risco.plot(range(len(fin_mensal)), fin_mensal["pct_anomalia"],
              color=COR_ALERTA, linewidth=2, marker="o", markersize=5,
              label="% Anomalia")

# Tendência do risco
if len(fin_mensal) > 2:
    slope_r, intercept_r, *_ = stats.linregress(
        range(len(fin_mensal)), fin_mensal["pct_anomalia"].fillna(0))
    tend_r = [intercept_r + slope_r * i for i in range(len(fin_mensal))]
    ax_risco.plot(range(len(fin_mensal)), tend_r, color=COR_ROXO,
                  linestyle=":", linewidth=2, alpha=0.8, label=f"Tend. risco ({slope_r:+.3f}%/mês)")

ax_vol.set_xticks(range(len(fin_mensal)))
ax_vol.set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
ax_vol.set_title("Volume de Pedidos × Taxa de Anomalia Mensal", fontsize=10)
ax_vol.set_ylabel("Nº de Pedidos", color=COR_RECEITA)
ax_risco.set_ylabel("% Anomalia", color=COR_ALERTA)
ax_risco.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax_risco.legend(loc="upper right", fontsize=8)
ax_vol.legend(loc="upper left", fontsize=8)

# Painel 2: Correlação scatter volume × % anomalia
corr_vr, p_vr = stats.pearsonr(
    fin_mensal["n_pedidos"].fillna(0),
    fin_mensal["pct_anomalia"].fillna(0)
)
axes[1].scatter(fin_mensal["n_pedidos"], fin_mensal["pct_anomalia"],
                color=COR_RECEITA, alpha=0.7, s=80)
if len(fin_mensal) > 2:
    z2 = np.polyfit(fin_mensal["n_pedidos"].fillna(0),
                    fin_mensal["pct_anomalia"].fillna(0), 1)
    p2 = np.poly1d(z2)
    x_line = np.linspace(fin_mensal["n_pedidos"].min(), fin_mensal["n_pedidos"].max(), 50)
    axes[1].plot(x_line, p2(x_line), color=COR_ALERTA, linestyle="--", linewidth=1.5)
for i, row in fin_mensal.iterrows():
    axes[1].annotate(row["ano_mes"], (row["n_pedidos"], row["pct_anomalia"]),
                     fontsize=6, alpha=0.6)
axes[1].set_title(f"Correlação: Volume × % Anomalia | r={corr_vr:.2f} | p={p_vr:.3f}", fontsize=10)
axes[1].set_xlabel("Pedidos no Mês")
axes[1].set_ylabel("% Anomalia de Pagamento")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))

plt.tight_layout()
salvar(fig, "02_risco_volume")
plt.show()

print(f"\n── Risco × Volume ──────────────────────")
print(f"  Correlação volume × anomalia : r={corr_vr:.2f} | p={p_vr:.3f}")
print(f"  Tendência da taxa de risco   : {slope_r:+.3f}%/mês")
print(f"  Interpretação: {'risco cresce com volume ⚠️' if corr_vr > 0.4 else '✅ crescimento não amplifica o risco'}")


---

## Análise 4 — Distribuição Estatística do Risco: É Precificável?

> *"Risco precificável é risco gerenciável. A distribuição estatística das taxas de anomalia mensal determina se o risco tem comportamento estável o suficiente para ser modelado com premissas confiáveis — condição necessária para que seja incorporado ao valuation sem exigir desconto excessivo por incerteza. Distribuições com cauda pesada indicam eventos extremos raros mas de alto impacto, que elevam o capital de reserva necessário e dificultam a precificação do risco residual."*

**Framework:** Controle Estatístico de Processo — análise de distribuição do risco  
**Entrega:** Distribuição estatística do risco com análise de cauda

**Como este script responde à pergunta:**
> O script analisa a distribuição estatística da taxa de anomalia mensal. Dois painéis respondem à pergunta:
>1. **Histograma da taxa de anomalia mensal:** Barras de frequência com linha vertical de média (vermelho) e limiar de +2 desvios padrão (roxo tracejado). A distância entre a média e o limiar de +2σ é a cauda de estresse — o cenário que o capital de reserva precisa aguentar. Se o histograma for assimétrico à direita (cauda longa), o risco tem eventos extremos raros mas muito custosos.
>2. **Box plot da taxa de anomalia:** Caixa com mediana (linha vermelha), bigodes e outliers (pontos). Um box plot largo indica alta variabilidade — risco difícil de precificar com premissas estáveis. Outliers acima do bigode superior são os meses em que o risco saiu do padrão esperado.

**Análise do Resultado:**
O coeficiente de variação (CV) é o indicador-síntese desta análise: quanto maior, mais imprevisível é o comportamento do risco ao longo do tempo e mais conservadoras precisam ser as premissas do modelo financeiro. Um histograma assimétrico à direita — cauda longa de eventos de alta anomalia — indica que o modelo precisa ser estressado além da média para capturar o custo real do risco herdado. O limiar de +2 desvios padrão define o cenário de estresse que o capital de reserva do comprador precisa suportar sem comprometer a operação.

In [ ]:
# ─── Distribuição estatística do risco ───────────────────────────────────────
anomalias_mensais = fin_mensal["pct_anomalia"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Bloco 2 — Distribuição Estatística do Risco de Crédito",
             fontsize=13, fontweight="bold")

# Painel 1: Histograma da taxa de anomalia
axes[0].hist(anomalias_mensais, bins=min(12, len(anomalias_mensais)),
             color=COR_RECEITA, alpha=0.7, edgecolor="white")
axes[0].axvline(anomalias_mensais.mean(), color=COR_ALERTA, linewidth=2,
                linestyle="--", label=f"Média: {anomalias_mensais.mean():.1f}%")
axes[0].axvline(anomalias_mensais.mean() + 2 * anomalias_mensais.std(),
                color=COR_ROXO, linewidth=1.5, linestyle=":",
                label=f"Cauda (+2σ): {anomalias_mensais.mean() + 2*anomalias_mensais.std():.1f}%")
axes[0].set_title("Distribuição da Taxa de Anomalia Mensal", fontsize=10)
axes[0].set_xlabel("% Anomalia Mensal")
axes[0].set_ylabel("Frequência")
axes[0].legend(fontsize=8)

# Painel 2: Box plot + estatísticas
axes[1].boxplot(anomalias_mensais, patch_artist=True,
                boxprops=dict(facecolor=COR_RECEITA, alpha=0.4, color=COR_RECEITA),
                medianprops=dict(color=COR_ALERTA, linewidth=2),
                whiskerprops=dict(color=COR_NEUTRO),
                capprops=dict(color=COR_NEUTRO),
                flierprops=dict(marker="o", color=COR_ALERTA, markersize=6))
axes[1].set_title("Box Plot — Taxa de Anomalia Mensal", fontsize=10)
axes[1].set_ylabel("% Anomalia")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xticklabels(["Taxa de Anomalia"])

plt.tight_layout()
salvar(fig, "02_distribuicao_risco")
plt.show()

cv_risco   = anomalias_mensais.std() / anomalias_mensais.mean() if anomalias_mensais.mean() > 0 else 0
cauda_2sig = anomalias_mensais.mean() + 2 * anomalias_mensais.std()

print("\n── Distribuição Estatística do Risco ─────────────────")
print(f"  Média   : {anomalias_mensais.mean():.2f}%")
print(f"  Mediana : {anomalias_mensais.median():.2f}%")
print(f"  Desvio  : {anomalias_mensais.std():.2f}%")
print(f"  CV      : {cv_risco:.2f} — {'alta variabilidade ⚠️' if cv_risco > 0.3 else '✅ variabilidade controlada'}")
print(f"  Cauda (+2σ): {cauda_2sig:.2f}% — cenário de estresse")
print(f"  Precificável: {'✅ distribuição estável' if cv_risco < 0.3 else '⚠️  alta variabilidade dificulta precificação'}")


---

## Análise 5 — Análise de Sensibilidade: Impacto de Choques de Risco

> *"A margem de segurança de um modelo financeiro é testada não pelo cenário esperado, mas pela sua capacidade de absorver perturbações. A simulação progressiva de choques sobre a taxa base de inadimplência quantifica em termos financeiros concretos o ponto de ruptura do modelo atual — o nível de deterioração do risco a partir do qual o spread capturado pelos intermediários é insuficiente para cobrir as perdas e o negócio passa a destruir valor."*

**Framework:** Matriz GUT — análise de sensibilidade do risco  
**Entrega:** Simulação de impacto de choques de inadimplência na margem operacional

**Como este script responde à pergunta:**
> O script simula o impacto financeiro de choques progressivos sobre a taxa base de inadimplência — 0%, +5%, +10%, +15%, +20% e +30%. Dois painéis respondem à pergunta:
>1. **Perda adicional por choque (R$ mil):** Barras coloridas em verde quando o spread ainda é positivo e em vermelho quando o choque elimina o spread. A mudança de cor marca visualmente o ponto de ruptura — o choque a partir do qual o spread é eliminado e o risco passa a destruir valor.
>2. **Spread líquido residual após choque:** Linha descendente com marcadores, linha de referência no zero e área verde sombreada onde o spread ainda é positivo. O cruzamento da linha com o zero é o limiar crítico: acima desse nível de inadimplência, o spread capturado é insuficiente para cobrir as perdas nas premissas atuais.

**Análise do Resultado:**
A mudança de cor das barras — de verde para vermelho — marca visualmente o threshold crítico: o choque a partir do qual o spread é eliminado e o risco passa a destruir valor líquido. Esse ponto de ruptura é o dado mais relevante para o comprador, pois define a margem de tolerância do modelo atual a deteriorações de crédito. Se o limiar crítico está próximo da taxa base observada, o ativo opera com baixa margem de segurança — qualquer aceleração de crescimento ou mudança no perfil de clientes pode cruzar esse limiar rapidamente. Se está distante, o modelo tem resiliência para absorver choques moderados sem comprometer a rentabilidade projetada.

In [ ]:
# ─── Sensibilidade: impacto de choques de inadimplência ──────────────────────
receita_total = fin_fato["preco"].sum()
spread_total  = fin_fato["spread_intermediario"].sum()
taxa_base     = fin_fato["anomalia_pagamento"].mean()

choques = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
impactos = []
for choque in choques:
    taxa_nova          = taxa_base * (1 + choque)
    perda_adicional    = receita_total * (taxa_nova - taxa_base)
    spread_liquido     = spread_total - perda_adicional
    margem_spread      = spread_liquido / receita_total * 100
    impactos.append({
        "choque_pct"     : choque * 100,
        "taxa_nova"      : taxa_nova * 100,
        "perda_adicional": perda_adicional,
        "spread_liquido" : spread_liquido,
        "margem_spread"  : margem_spread,
    })

df_sens = pd.DataFrame(impactos)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Bloco 2 — Análise de Sensibilidade: Impacto de Choques de Risco",
             fontsize=13, fontweight="bold")

# Painel 1: Perda adicional por choque
cores_sens = [COR_MARGEM if sl > 0 else COR_ALERTA for sl in df_sens["spread_liquido"]]
axes[0].bar([f"+{c:.0f}%" for c in df_sens["choque_pct"]], df_sens["perda_adicional"] / 1000,
            color=cores_sens, alpha=0.85)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("Perda Adicional por Choque de Inadimplência (R$ mil)", fontsize=10)
axes[0].set_xlabel("Choque (% sobre taxa base)")
axes[0].set_ylabel("Perda Adicional (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))

# Painel 2: Spread líquido após choque
axes[1].plot([f"+{c:.0f}%" for c in df_sens["choque_pct"]], df_sens["spread_liquido"] / 1000,
             color=COR_RECEITA, linewidth=2, marker="o", markersize=8)
axes[1].axhline(0, color=COR_ALERTA, linestyle="--", linewidth=1.5, alpha=0.7)
axes[1].fill_between([f"+{c:.0f}%" for c in df_sens["choque_pct"]],
                     [0] * len(df_sens),
                     df_sens["spread_liquido"] / 1000,
                     where=df_sens["spread_liquido"] > 0,
                     alpha=0.15, color=COR_MARGEM, label="Zona positiva")
axes[1].set_title("Spread Líquido Residual Após Choque (R$ mil)", fontsize=10)
axes[1].set_xlabel("Choque (% sobre taxa base)")
axes[1].set_ylabel("Spread Líquido (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))

plt.tight_layout()
salvar(fig, "02_sensibilidade_risco")
plt.show()

print("\n── Tabela de Sensibilidade ─────────────────────────")
print(f"  Taxa base de anomalia: {taxa_base*100:.2f}%")
print(f"  Spread base total    : R$ {spread_total:,.0f}")
print()
print(df_sens[["choque_pct", "taxa_nova", "perda_adicional", "spread_liquido", "margem_spread"]]
      .rename(columns={"choque_pct":"Choque%","taxa_nova":"Taxa Nova%",
                       "perda_adicional":"Perda (R$)","spread_liquido":"Spread Líq (R$)",
                       "margem_spread":"Margem%"})
      .to_string(index=False, float_format="{:,.0f}".format))

ponto_ruptura = df_sens[df_sens["spread_liquido"] < 0]["choque_pct"].min() if any(df_sens["spread_liquido"] < 0) else None
if ponto_ruptura:
    print(f"\n  ⚠️  Ponto de ruptura: spread vira negativo com choque de +{ponto_ruptura:.0f}%")
else:
    print(f"\n  ✅ Spread permanece positivo mesmo nos cenários de estresse simulados")


In [ ]:
# ─── Recálculo defensivo — garante que as variáveis existam ─────────────────
# (necessário se a célula for executada fora de ordem)
try:
    _ = cv_risco
except NameError:
    _anomalias_tmp = fin_mensal["pct_anomalia"].dropna()
    cv_risco = (_anomalias_tmp.std() / _anomalias_tmp.mean()
                if _anomalias_tmp.mean() > 0 else 0)
try:
    _ = corr_vr
except NameError:
    from scipy import stats as _stats
    corr_vr, _ = _stats.pearsonr(
        fin_mensal["n_pedidos"].fillna(0),
        fin_mensal["pct_anomalia"].fillna(0)
    )
try:
    _ = cauda_2sig
except NameError:
    _a = fin_mensal["pct_anomalia"].dropna()
    cauda_2sig = _a.mean() + 2 * _a.std()

# ─── Score do Bloco 2 ────────────────────────────────────────────────────────
taxa_anomalia_geral = fin_fato["anomalia_pagamento"].mean() * 100
_b2_risco_alto     = taxa_anomalia_geral > 15
_b2_cv_alto        = cv_risco > 0.3
_b2_correlacao     = corr_vr > 0.4

if _b2_risco_alto and _b2_cv_alto:
    _b2_score = 1
    _b2_sinal = "🔴 Risco elevado e volátil — exposição financeira relevante para o comprador"
    _b2_cor   = COR_ALERTA
elif _b2_risco_alto or _b2_cv_alto:
    _b2_score = 2
    _b2_sinal = "⚠️  Risco moderado — requer monitoramento rigoroso no pós-aquisição"
    _b2_cor   = COR_DESTAQUE
else:
    _b2_score = 3
    _b2_sinal = "✅ Risco controlado e previsível — perfil financeiro favorável para o comprador"
    _b2_cor   = COR_MARGEM

_b2_cond = (
    f"Estabelecer capital de reserva de pelo menos {cauda_2sig:.1f}% da receita "
    f"como proteção ao risco de crédito (cenário de estresse +2σ)."
    if _b2_score <= 2
    else "Perfil de risco compatível com a tese de aquisição."
)

print("=" * 60)
print("SÍNTESE — BLOCO 2: INADIMPLÊNCIA E EXPOSIÇÃO AO RISCO")
print("=" * 60)
print(f"  Taxa de anomalia geral     : {taxa_anomalia_geral:.2f}%")
print(f"  CV do risco               : {cv_risco:.2f}")
print(f"  Correlação volume × risco : r={corr_vr:.2f}")
print(f"  Cauda de estresse (+2σ)   : {cauda_2sig:.2f}%")
if ponto_ruptura:
    print(f"  Ponto de ruptura          : choque de +{ponto_ruptura:.0f}% sobre a taxa base")
else:
    print(f"  Ponto de ruptura          : não identificado nos cenários simulados")
print()
print(f"  Score do Bloco: {_b2_score}/3")
print(f"  Sinal        : {_b2_sinal}")
print()
print(f"  Condicionante: {_b2_cond}")
print("=" * 60)


---
*Próximo notebook: `03_rentabilidade_financeira.ipynb` — O spread capturado pelos intermediários financeiros representa uma oportunidade de valor para o comprador — ou o risco associado não justifica a exposição?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
